# Sheet 3.2: PPO and RL for LLMs

**Roadmap topic 3.2** (Fanout 07.04). Read first: the PPO and GAE papers, *The 37 Implementation Details of PPO*, InstructGPT section 3, *The N Implementation Details of RLHF with PPO*, and RLHF Book chapters on policy gradients.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. CPU is fine (Exercise 3 trains PPO on CartPole in a few minutes).

In [ ]:
import math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym
torch.manual_seed(0)

## Exercise 1: Generalised Advantage Estimation

`gae(rewards, values, dones, last_value, gamma, lam)` for one environment's rollout of length `T`:
`δ_t = r_t + γ·V(s_{t+1})·(1 - done_t) - V(s_t)`, `A_t = δ_t + γλ·(1 - done_t)·A_{t+1}`, with `V(s_T) = last_value`. Return `(advantages, returns)` where `returns = advantages + values`.

First compute the answer for the check's 5-step example **by hand** in the markdown cell below, then implement.

In [ ]:
def gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    # YOUR CODE
    raise NotImplementedError

**By hand:** `r = [1, 1, 1, 0, 2]`, `V = [0.5, 0.4, 0.3, 0.2, 0.1]`, `done = [0, 0, 1, 0, 0]`, `last_value = 1.0`, `γ = 0.9`, `λ = 0.5`. Write `A_0 … A_4`:

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    r = torch.tensor([1., 1., 1., 0., 2.]); V = torch.tensor([.5, .4, .3, .2, .1]); d = torch.tensor([0., 0., 1., 0., 0.])
    A, R = gae(r, V, d, 1.0, 0.9, 0.5)
    d4 = 2 + 0.9 * 1.0 - 0.1; d3 = 0 + 0.9 * 0.1 - 0.2; d2 = 1 - 0.3; d1 = 1 + 0.9 * 0.3 - 0.4; d0 = 1 + 0.9 * 0.4 - 0.5
    A4 = d4; A3 = d3 + 0.45 * A4; A2 = d2; A1 = d1 + 0.45 * A2; A0 = d0 + 0.45 * A1
    assert torch.allclose(A, torch.tensor([A0, A1, A2, A3, A4]), atol=1e-6), A
    assert torch.allclose(R, A + V)
    A1_, _ = gae(r, V, d, 1.0, 0.9, 1.0); mc = torch.tensor([1 + .9 + .81, 1 + .9, 1, 0 + .9 * 2 + .81 * 1, 2 + .9]) - V
    assert torch.allclose(A1_, mc, atol=1e-6), "lambda=1 must give Monte-Carlo returns minus V"
    print("PASS")
_check1()

## Exercise 2: The PPO loss and its diagnostics

`ppo_loss(new_logp, old_logp, adv, values, old_values, returns, entropy, clip=0.2, vf_clip=0.2, vf_coef=0.5, ent_coef=0.01)` returns `(loss, stats)`:
- policy loss: `-mean(min(ρ·A, clip(ρ, 1-ε, 1+ε)·A))`, with `ρ = exp(new - old)`
- value loss: CleanRL-style clipped value loss `0.5·mean(max((V-R)², (V_old + clip(V-V_old, ±vf_clip) - R)²))`
- total: `pg + vf_coef·v - ent_coef·mean(entropy)`
- `stats`: `approx_kl` (the low-variance `k3` estimator: `mean((ρ-1) - log ρ)`), `clipfrac` (fraction of samples where `|ρ-1| > ε`).
Advantages are passed in already normalised.

In [ ]:
def ppo_loss(new_logp, old_logp, adv, values, old_values, returns, entropy, clip=0.2, vf_clip=0.2, vf_coef=0.5, ent_coef=0.01):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    old = torch.zeros(4); adv = torch.tensor([1., 1., -1., -1.])
    new = torch.log(torch.tensor([1.5, 0.9, 0.5, 1.1])).requires_grad_()
    z = torch.zeros(4); loss, st = ppo_loss(new, old, adv, z, z, z, z)
    assert abs(loss.item() - (-(1.2 + 0.9 - 0.8 - 1.1) / 4)) < 1e-6, "policy loss wrong"
    loss.backward()
    assert new.grad[0] == 0 and new.grad[2] == 0, "clipped samples must give zero gradient"
    assert new.grad[1] != 0 and new.grad[3] != 0
    assert abs(st["clipfrac"] - 0.5) < 1e-9
    r = torch.tensor([1.5, 0.9, 0.5, 1.1]); assert abs(st["approx_kl"] - float(((r - 1) - r.log()).mean())) < 1e-6
    v = torch.tensor([1.0, 0.0]); vo = torch.tensor([0.0, 0.0]); R = torch.tensor([1.0, 1.0])
    _, s2 = ppo_loss(torch.zeros(2), torch.zeros(2), torch.zeros(2), v, vo, R, torch.zeros(2))
    assert abs(s2["v"] - 0.5 * ((0.8 ** 2) + 1.0) / 2) < 1e-6, "clipped value loss wrong"
    print("PASS")
_check2()

## Exercise 3: PPO from scratch on CartPole

`ppo_cartpole(total_steps, seed)`: 4 vectorised envs (`gym.vector.SyncVectorEnv`), 128 steps per env per rollout, GAE(0.99, 0.95), 4 epochs × 4 minibatches, Adam 2.5e-4 with linear LR decay, advantage normalisation per minibatch, gradient clipping 0.5, separate actor and critic MLPs (64-64 tanh, orthogonal init). Use `autoreset_mode=gym.vector.AutoresetMode.SAME_STEP` so `info['final_obs']` holds the last observation of a finished episode, and handle `terminated` vs `truncated` correctly (bootstrap through truncation). Return the list of finished-episode returns and a list of `stats` dicts per update.

Target: 100-episode average ≥ 450 within 300k steps (a few minutes on CPU). Most failures here come from one of the 37 implementation details, so reread them when stuck.

In [ ]:
def ppo_cartpole(total_steps=300_000, seed=0):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    t0 = time.time(); rets, stats = ppo_cartpole(300_000, seed=0)
    assert len(rets) > 100, "too few finished episodes recorded; are you collecting returns?"
    best = max(np.mean(rets[i:i + 100]) for i in range(len(rets) - 100))
    assert best >= 450, f"best 100-episode average {best:.0f} < 450"
    kls = [s["approx_kl"] for s in stats]
    assert np.median(kls) < 0.05, f"median approx KL {np.median(kls):.3f} is large; check ratios and learning rate"
    print(f"PASS  best 100-ep avg {best:.0f} in {time.time() - t0:.0f}s, median approx_kl {np.median(kls):.1e}")
_check3()

## Exercise 4: Break it on purpose

Make three broken variants of your PPO and record `approx_kl`, `clipfrac`, explained variance (`1 - Var(R - V)/Var(R)`) and returns:
1. learning rate × 20
2. no advantage normalisation and rewards × 100
3. 30 epochs per update instead of 4

Fill in a table mapping each failure to its signature in the metrics. This table is what you'll use when an LLM RL run misbehaves.

### On Colab (optional): LunarLander-v3 and continuous control
Extend to `LunarLanderContinuous-v3` with a Gaussian policy (state-independent log-std). Compare with CleanRL's `ppo_continuous_action.py` curve over 3 seeds.

In [ ]:
# Your break-it experiments and table

## Checkpoint
With equations, explain: the importance ratio and why PPO clips it, why clipping alone doesn't bound the KL (and what `approx_kl` early-stopping does), and where the KL penalty to the reference model enters in RLHF (InstructGPT eq. 2: inside the reward, per token).